This notebook needs Pat Phoompuang's library files

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('..', 'YT-Simulation')))
from library.prepare_data import prepare_simulation_data
from library.continuum_grid import compute_continuum_grid
from library.yt_fields import add_flux_fields

import numpy as np
import scipy
import copy

In [3]:
data_file = os.path.abspath(os.path.join('.', 'data/output_00273/info_00273.txt'))
filter_files = os.path.abspath(os.path.join('.', 'silmaril/silmaril/data/mean_throughputs'))
filters = ['F070W', 'F090W', 'F115W', 'F140M', 'F150W', 'F150W', 'F162M', 'F164N', 'F182M', 'F187N', 'F200W', 'F210M', 'F212N', 'F250M', 'F277W', 'F300M', 'F322W2', 'F323N', 'F335M', 'F356W', 'F360M', 'F405N', 'F410M', 'F430M', 'F444W', 'F460M', 'F466N', 'F470N', 'F480M', 'WLP4']
filter = filters[4]
filter_file = os.path.join(filter_files, filter + r"_mean_system_throughput.txt")

In [4]:
ds, ad, ctr, pop2, wl_shifted, trans, width = prepare_simulation_data(
    input_path = data_file,
    filter_path = filter_file
)

yt : [INFO     ] 2026-03-09 16:08:21,307 Parameters: current_time              = 0.3604448649237178 Gyr
yt : [INFO     ] 2026-03-09 16:08:21,308 Parameters: domain_dimensions         = [64 64 64]
yt : [INFO     ] 2026-03-09 16:08:21,311 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 16:08:21,312 Parameters: domain_right_edge         = [1. 1. 1.]
yt : [INFO     ] 2026-03-09 16:08:21,315 Parameters: cosmological_simulation   = True
yt : [INFO     ] 2026-03-09 16:08:21,317 Parameters: current_redshift          = 12.171087046255657
yt : [INFO     ] 2026-03-09 16:08:21,317 Parameters: omega_lambda              = 0.685000002384186
yt : [INFO     ] 2026-03-09 16:08:21,318 Parameters: omega_matter              = 0.314999997615814
yt : [INFO     ] 2026-03-09 16:08:21,321 Parameters: omega_radiation           = 0.0
yt : [INFO     ] 2026-03-09 16:08:21,322 Parameters: hubble_constant           = 0.674000015258789
yt : [WARNING  ] 2026-03-09 16:08:21,515 This output

Loading filter data from file: c:\Users\kevin\aether\silmaril\silmaril\data\mean_throughputs\F150W_mean_system_throughput.txt
Filter wavelength range: 948.1-1330.0 Å


In [7]:
df_results, interp_funcs = compute_continuum_grid(
    min_temp = 1e3, max_temp = 1e5, num_temp_grid = 15,
    min_dense = 1e-4, max_dense = 1e6, num_dense_grid = 15,
    min_wl = 912, max_wl = 1e5, num_wl_grid = 10000,
    filter_wl = wl_shifted,     # 1D array from prepare_simulation_data
    filter_output = trans,      # 1D array from prepare_simulation_data
    save_dir = "continuum_single"
)

Processing single filter...
Saving dataframe to: continuum_single
Filter range: 949.6-1328.5 Å
Points with transmission > 0: 500


c:\Users\kevin\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyneb\core\continuum.py:135: RuntimeWarning: invalid value encountered in power
  A = 202.0 * (y * (1. - y) * (1. -(4. * y * (1 - y))**0.8) + 0.88 * ( y * (1 - y))**1.53 * (4. * y * (1 - y))**0.8)


Saved: df_wl950-1329A.txt (225 rows)
Creating variable df01 for single filter:
  Created df01 = shape (225, 5)


In [9]:
# add flux fields to yt dataset
ds, filter_list = add_flux_fields(
    ds = ds, 
    interp_funcs = interp_funcs, 
    min_temp = 1e3, 
    max_temp = 1e5, 
    min_dense = 1e-4, 
    max_dense = 1e6,
    he_h_ratio = 0.1
)

Processing single filter...
Added: flux_contH
Added: flux_cont2p
Added: flux_contff
Added: flux_total

Successfully added fields for single filter


In [12]:
from merlin_spectra.initializer import Initializer
from merlin_spectra.constants import default_lines as lines, default_wavelengths as wavelengths

In [13]:
merlin = Initializer(ds)
merlin._load_fields()
ds = merlin.ds
merlin._load_luminosity_flux()
ad = ds.all_data()

TypeError: expected str, bytes or os.PathLike object, not RAMSESDataset

In [ ]:
def get_filter_interpolator(
    filter_file: str,
    z: float = 0.0,
):
    """
    Returns a CubicSpline interpolator for a given JWST filter
    based on the starburst spectrum table.

    Parameters
    ----------
    filter_file : str
        Absolute or relative path to filter throughput file
    z : float
        Redshift of the galaxy (default 0)

    Returns
    -------
    interpolator : scipy.interpolate.CubicSpline
        Function of stellar age [Myr] giving mean photon-rate-weighted flux
        through the filter.
    """
    # Load filter throughput
    filter_data = np.loadtxt(filter_file, skiprows=1)
    wav_angs = filter_data[:, 0] * 1e4  # microns -> angstroms, rest-frame
    
    # Return CubicSpline interpolator
    return wav_angs, scipy.interpolate.CubicSpline(wav_angs, filter_data[:, 1])


In [ ]:
def get_filter_luminosity(filter):
    filter_file = os.path.join(filter_files, filter + r"_mean_system_throughput.txt")
    z=ds.current_redshift
    angs, interp = get_filter_interpolator(filter_file, z=z)

    def _filter_lum(field, data):
        """
        Sum of line luminosities weighted by the filter interpolator
        """
        lum_sum = np.zeros_like(data['gas', 'luminosity_' + lines[0]])
        for line, lam in zip(lines, wavelengths):
            shifted_lam = lam * (1+z)
            weight = interp(shifted_lam)  # filter weight for this line
            weight = np.where(shifted_lam >= angs[0] and shifted_lam <= angs[-1], weight, 0)
            if weight >= 1e-6:
                lum_sum += data['gas', 'luminosity_' + line] * weight
        return lum_sum

    return copy.deepcopy(_filter_lum)

In [ ]:
for f in filters:
    ds.add_field(
        ("gas", "lum_filter_" + f),  # field name
        function=get_filter_luminosity(f),
        units='cm**3',          # adjust if your line luminosities have different units
        sampling_type="cell",
        force_override=True
    )